# Data Cleaning 

This notebook cleans the raw data available in data/raw and writes the clean version back to the folder data/processed. 

In [26]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from c08_farming_exit import config, features, data_cleaning, mappings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

In [2]:
COUNTRIES = {
    "Botswana": config.RAW_DATA_DIR / "Botswana",
    "Kenya":    config.RAW_DATA_DIR / "Kenya",
    "Namibia":  config.RAW_DATA_DIR / "Namibia",
    "Tanzania": config.RAW_DATA_DIR / "Tanzania",
    "Zambia":   config.RAW_DATA_DIR / "Zambia",
}

## 1. Database

In [10]:
#THE DATABASE CONSISTS OF ADULTS ONLY
database = []

for country, base_path in COUNTRIES.items():
    identifying_info    = data_cleaning.load_and_preprocess(base_path, f"{country}_identifying_info.csv",                   features.IDENTIFYING_INFO_2023)
    hh_members          = data_cleaning.load_and_preprocess(base_path, f"{country}_household_members_characterstics.csv",   features.HH_MEMBERS_2023)

    merge = identifying_info.merge(hh_members, on=["interview_key"], how="inner")
    
    filtered = merge[
        merge["relation_to_head"].isin([    "Self/Head", 
                                            "Wife/Husband", 
                                            "Son/Daughter-In-Law", 
                                            "Sister/Brother", 
                                            "Mother/Father", 
                                            "Brother/Sister-In-Law", 
                                            "Grandfather/Mother", 
                                            "Father/Mother-In-Law"]) &
                                        (merge["age"] >= 18)
    ]

    database.append(filtered)

df_database = pd.concat(database, ignore_index=True)

#CREATE PERSONAL IDENTIFIER 
df_database["personal_id"] = df_database["country"] + "_" + df_database["interview_key"].astype(str) + "_" + df_database["members_id"].astype(str)

#FINAL SORTING
df_database = df_database.query("country != 'YES+A112:L126+A112:C126'")

[Botswana_identifying_info.csv] Missing columns: ['ea', 'region']
[Namibia_identifying_info.csv] Missing columns: ['dist']


## 2. Features

### 2.1 Creating HH-Level Features

In [20]:
hh_features = []

for country, base_path in COUNTRIES.items():
    #ONE OBSERVATION PER HH
    land_ownership          = data_cleaning.load_and_preprocess(base_path, f"{country}_land_ownership_and_access.csv",         features.LAND_OWNERSHIP_ACCESS_2023)
    crop_expenditure        = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_on_crops.csv",              features.CROP_EXPENDITURE_2023)
    lifestock_grazing       = data_cleaning.load_and_preprocess(base_path, f"{country}_grazing_patterns_and_schemes.csv",      features.LIFESTOCK_GRAZING_2023)
    livestock_income        = data_cleaning.load_and_preprocess(base_path, f"{country}_income_livestock.csv",                  features.LIFESTOCK_INCOME_2023)
    livestock_expenditure   = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_livestock.csv",             features.LIFESTOCK_EXPENDITURE_2023)
    housing_conditions      = data_cleaning.load_and_preprocess(base_path, f"{country}_housing_conditions.csv",                features.HOUSING_CONDITIONS_2023)
    energy_access           = data_cleaning.load_and_preprocess(base_path, f"{country}_access_to_energy.csv",                  features.ENERGY_ACCESS_2023)
    internet_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_internet_access.csv",                   features.INTERNET_ACCESS_2023)
    social_network          = data_cleaning.load_and_preprocess(base_path, f"{country}_other_household_social_network.csv",    features.SOCIAL_NETWORK_2023)
    social_embeddedness     = data_cleaning.load_and_preprocess(base_path, f"{country}_social_embeddedness.csv",               features.SOCIAL_EMBEDDEDNESS_2023)
    food_insecurity         = data_cleaning.load_and_preprocess(base_path, f"{country}_food_insecurity_experiance_scale.csv",  features.FOOD_INSECURITY_2023)
    road_connectivity       = data_cleaning.load_and_preprocess(base_path, f"{country}_road_connectivity.csv",                 features.ROAD_CONNECTIVITY_2023)
    
    land_ownership          = data_cleaning.convert_land_sizes_to_acres(land_ownership, country)

    #MANY OBSERVATIONS PER HH
    if country == "Tanzania":
        #Tanzania's market access data is stored in value_chains.csv.
        market_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_vale_chains.csv",                       features.MARKET_ACCESS_2023)
    else:
        market_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_market_access.csv",                     features.MARKET_ACCESS_2023)
    crop_production           = data_cleaning.load_and_preprocess(base_path, f"{country}_crop_production.csv",                   features.CROP_PRODUCTION_2023)
    livestock_ownership       = data_cleaning.load_and_preprocess(base_path, f"{country}_livestock_ownership.csv",               features.LIVESTOCK_OWNERSHIP_2023)
    assets_owned              = data_cleaning.load_and_preprocess(base_path, f"{country}_assets.csv",                            features.ASSETS_OWNED_2023)
    other_income              = data_cleaning.load_and_preprocess(base_path, f"{country}_other_income.csv",                      features.OTHER_INCOME_SOURCES_2023)
    shocks_and_coping         = data_cleaning.load_and_preprocess(base_path, f"{country}_shocks_and_coping.csv",                 features.SHOCKS_AND_COPING_2023)

    market_access             = data_cleaning.resolve_duplicates(market_access, key_col="interview_key", sort_col="crop_contract_crop_type", ascending=True)
    crop_production           = data_cleaning.create_crop_production_features(crop_production)
    livestock_ownership       = data_cleaning.create_livestock_features(livestock_ownership)
    assets_owned              = data_cleaning.create_asset_features(assets_owned)
    other_income              = data_cleaning.create_other_income_features(other_income, country)
    shocks                    = data_cleaning.create_shock_features(shocks_and_coping)
    coping                    = data_cleaning.create_coping_features(shocks_and_coping)

    merges = [
        (land_ownership,          ["interview_key"],     "outer"),
        (crop_expenditure,        ["interview_key"],     "outer"),
        (lifestock_grazing,       ["interview_key"],     "outer"),
        (livestock_income,        ["interview_key"],     "outer"),
        (livestock_expenditure,   ["interview_key"],     "outer"),
        (housing_conditions,      ["interview_key"],     "outer"),
        (energy_access,           ["interview_key"],     "outer"),
        (internet_access,         ["interview_key"],     "outer"),
        (social_network,          ["interview_key"],     "outer"),
        (social_embeddedness,     ["interview_key"],     "outer"),
        (food_insecurity,         ["interview_key"],     "outer"),
        (road_connectivity,       ["interview_key"],     "outer"),
        (market_access,           ["interview_key"],     "outer"),
        (crop_production,         ["interview_key"],     "outer"),
        (livestock_ownership,     ["interview_key"],     "outer"),
        (assets_owned,            ["interview_key"],     "outer"),
        (other_income,            ["interview_key"],     "outer"),
        (shocks,                  ["interview_key"],     "outer"),
        (coping,                  ["interview_key"],     "outer"),
    ]

    df_help = None

    for df, keys, how in merges:
        if df_help is None:
            df_help = df
        else:
            df_help = df_help.merge(df, on=keys, how=how)
                
    #Adding the country to the table for identification
    df_help.insert(0, "country", country)

    hh_features.append(df_help)

df_hh_features = pd.concat(hh_features, ignore_index=True)



convert_land_sizes_to_acres: Botswana - Dropped 8 rows with NaN in 'land_measurement'
create_other_income_features: Botswana - Dropped 11 rows with NaN in 'other_income_frequency'
convert_land_sizes_to_acres: Namibia - Dropped 3 rows with NaN in 'land_measurement'
create_other_income_features: Namibia - Dropped 15 rows with NaN in 'other_income_frequency'
convert_land_sizes_to_acres: Tanzania - Dropped 1 rows with NaN in 'land_measurement'
[Tanzania_vale_chains.csv] Missing columns: ['markt_buyer']
convert_land_sizes_to_acres: Zambia - Dropped 144 rows with NaN in 'land_measurement'
create_other_income_features: Zambia - Dropped 2 rows with NaN in 'other_income_frequency'


In [6]:
# missing_by_country = (
#     df_hh_features
#     .groupby('country')
#     .apply(lambda g: g.isna().mean())
#     .sort_index()
# )
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     display(missing_by_country)


### 2.2 Creating Individual-Level Features

In [ ]:
individual_features = []

for country, base_path in COUNTRIES.items():
    #MANY OBSERVATIONS PER PERSON
    off_farm_employment    = data_cleaning.load_and_preprocess(base_path, f"{country}_off_farm_employment.csv",         features.OFF_FARM_EMPLOYMENT_2023)
    off_farm_employment    = data_cleaning.create_off_farm_employment_features(off_farm_employment)


    merges = [
            (off_farm_employment,  ["interview_key", "members_id"],     "outer"),
        ]

    df_help = None

    for df, keys, how in merges:
        if df_help is None:
            df_help = df
        else:
            df_help = df_help.merge(df, on=keys, how=how)
                
    #Adding the country to the table for identification
    df_help.insert(0, "country", country)

    individual_features.append(df_help)

df_individual_features = pd.concat(individual_features, ignore_index=True)

#CREATE PERSONAL IDENTIFIER 
df_individual_features["personal_id"] = df_individual_features["country"] + "_" + df_individual_features["interview_key"].astype(str) + "_" + df_individual_features["members_id"].astype(str)

## 3. Write clean data to data/raw folder

In [ ]:
df.to_csv(config.PROCESSED_DATA_DIR / "clean_data.csv", index=False)